In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

home_credit_credit_risk_model_stability_path = kagglehub.competition_download('home-credit-credit-risk-model-stability')
shivamagarwal29_credit_risk_path = kagglehub.notebook_output_download('shivamagarwal29/credit-risk')

print('Data source import complete.')


In [ ]:
import os
import zipfile
import pandas as pd
import sqlite3

print("Inspecionando os arquivos e diretórios no Kaggle...")
conexao = sqlite3.connect('banco_risco_credito.db')

caminho_arquivo = ""

# 1. Varrendo o diretório para encontrar arquivos úteis ou descompactar zips se necessário
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        caminho_completo = os.path.join(dirname, filename)
        print(f"Encontrado: {filename}")

        # Se for um arquivo zip, vamos extraí-lo automaticamente para a pasta de trabalho
        if filename.lower().endswith('.zip'):
            print(f"-> Descompactando o arquivo ZIP: {filename}")
            with zipfile.ZipFile(caminho_completo, 'r') as zip_ref:
                zip_ref.extractall('/kaggle/working/')

        elif filename.lower().endswith(('.csv', '.xlsx', '.xls')):
            caminho_arquivo = caminho_completo

# 2. Se encontrou direto na pasta input
if caminho_arquivo:
    print(f"-> Arquivo de dados localizado: {caminho_arquivo}")
else:
    # Procurar na pasta de trabalho caso tenha sido descompactado do zip
    for dirname, _, filenames in os.walk('/kaggle/working'):
        for filename in filenames:
            if filename.lower().endswith(('.csv', '.xlsx', '.xls')) and 'banco_risco_credito' not in filename:
                caminho_arquivo = os.path.join(dirname, filename)
                print(f"-> Arquivo localizado após descompactação: {caminho_arquivo}")
                break

if not caminho_arquivo:
    raise FileNotFoundError("Nenhum arquivo de dados (.csv ou .xlsx) foi encontrado. Certifique-se de adicionar um dataset de crédito em '+ Add Input' no canto superior direito.")

# 3. Leitura inteligente dependendo da extensão
if caminho_arquivo.lower().endswith('.csv'):
    df = pd.read_csv(caminho_arquivo)
else:
    df = pd.read_excel(caminho_arquivo)

# 4. Criando a tabela transacional no SQLite
df.to_sql('tb_credito', conexao, if_exists='replace', index=False)

print(f"\nSucesso! Tabela 'tb_credito' criada com {df.shape[0]} linhas e {df.shape[1]} colunas.")



Inspecionando os arquivos e diretórios no Kaggle...
Encontrado: sample_submission.csv
Encontrado: feature_definitions.csv
Encontrado: test_deposit_1.parquet
Encontrado: test_applprev_2.parquet
Encontrado: test_static_cb_0.parquet
Encontrado: test_static_0_0.parquet
Encontrado: test_credit_bureau_a_1_3.parquet
Encontrado: test_credit_bureau_a_1_2.parquet
Encontrado: test_tax_registry_b_1.parquet
Encontrado: test_static_0_2.parquet
Encontrado: test_credit_bureau_a_2_3.parquet
Encontrado: test_credit_bureau_a_2_9.parquet
Encontrado: test_debitcard_1.parquet
Encontrado: test_credit_bureau_a_1_1.parquet
Encontrado: test_credit_bureau_a_2_2.parquet
Encontrado: test_credit_bureau_a_2_11.parquet
Encontrado: test_applprev_1_2.parquet
Encontrado: test_credit_bureau_a_2_1.parquet
Encontrado: test_credit_bureau_a_1_4.parquet
Encontrado: test_tax_registry_c_1.parquet
Encontrado: test_applprev_1_0.parquet
Encontrado: test_tax_registry_a_1.parquet
Encontrado: test_credit_bureau_a_2_6.parquet
Encontra

In [ ]:
import pandas as pd
import sqlite3

# Conectando ao nosso banco de dados SQLite
conexao = sqlite3.connect('banco_risco_credito.db')

# Consulta SQL para inspecionar a estrutura e amostra dos dados
query_eda = """
    SELECT *
    FROM tb_credito
    LIMIT 5;
"""

# Trazendo para o Pandas e exibindo na tela
df_amostra = pd.read_sql(query_eda, conexao)
print("Amostra dos dados de crédito (Top 5 linhas):")
display(df_amostra)


Amostra dos dados de crédito (Top 5 linhas):


,case_id,last180dayaveragebalance_704A,last180dayturnover_1134A,last30dayturnover_651A,num_group1,openingdate_857D
0,225,None,None,None,0,2016-08-16
1,331,None,None,None,0,2015-03-19
2,358,None,None,None,0,2014-09-02
3,390,None,None,None,0,2014-07-23
4,390,None,None,None,1,2015-10-01


In [ ]:
import pandas as pd
import sqlite3

conexao = sqlite3.connect('banco_risco_credito.db')

# Consulta SQL para verificar contagem total de registros e valores nulos nas principais colunas
query_nulos = """
    SELECT
        COUNT(*) as Total_Registros,
        SUM(CASE WHEN last180dayaveragebalance_704A IS NULL THEN 1 ELSE 0 END) as Nulos_Saldo_180d,
        SUM(CASE WHEN last180dayturnover_1134A IS NULL THEN 1 ELSE 0 END) as Nulos_Turnover_180d,
        SUM(CASE WHEN openingdate_857D IS NULL THEN 1 ELSE 0 END) as Nulos_Data_Abertura
    FROM tb_credito;
"""

df_nulos = pd.read_sql(query_nulos, conexao)
print("Diagnóstico de Valores Nulos na Base de Crédito:")
display(df_nulos)


Diagnóstico de Valores Nulos na Base de Crédito:


,Total_Registros,Nulos_Saldo_180d,Nulos_Turnover_180d,Nulos_Data_Abertura
0,157302,145086,146221,12711


In [ ]:
import pandas as pd
import sqlite3

conexao = sqlite3.connect('banco_risco_credito.db')

# Lendo a tabela do SQLite para o Pandas para realizar o tratamento avançado
df_credito = pd.read_sql("SELECT * FROM tb_credito", conexao)

# 1. Tratamento de Nulos nas colunas financeiras (Substituindo NaN por 0, indicando ausência de movimentação no período)
colunas_financeiras = ['last180dayaveragebalance_704A', 'last180dayturnover_1134A', 'last30dayturnover_651A']
for col in colunas_financeiras:
    if col in df_credito.columns:
        df_credito[col] = df_credito[col].fillna(0)

# 2. Convertendo a data de abertura para formato datetime e preenchendo nulos com uma data padrão antiga
if 'openingdate_857D' in df_credito.columns:
    df_credito['openingdate_857D'] = pd.to_datetime(df_credito['openingdate_857D'], errors='coerce')
    df_credito['openingdate_857D'] = df_credito['openingdate_857D'].fillna(pd.to_datetime('2000-01-01'))

# 3. Salvando a tabela tratada de volta no banco SQLite
df_credito.to_sql('tb_credito_tratada', conexao, if_exists='replace', index=False)

print("Tratamento concluído com sucesso!")
print(f"Nova tabela 'tb_credito_tratada' salva com {df_credito.shape[0]} linhas e {df_credito.shape[1]} colunas.")
print(f"Valores nulos restantes nas colunas financeiras: {df_credito[colunas_financeiras].isnull().sum().sum()}")


Tratamento concluído com sucesso!
Nova tabela 'tb_credito_tratada' salva com 157302 linhas e 6 colunas.
Valores nulos restantes nas colunas financeiras: 0


In [ ]:
import pandas as pd
import sqlite3

conexao = sqlite3.connect('banco_risco_credito.db')

# Consulta SQL para agregar os dados por cliente (case_id)
query_agregada = """
    SELECT
        case_id,
        COUNT(case_id) AS Qtd_Transacoes_Registradas,
        MAX(last180dayaveragebalance_704A) AS Max_Saldo_180d,
        SUM(last180dayturnover_1134A) AS Soma_Turnover_180d,
        SUM(last30dayturnover_651A) AS Soma_Turnover_30d,
        MIN(openingdate_857D) AS Data_Primeira_Abertura
    FROM tb_credito_tratada
    GROUP BY case_id
    ORDER BY Soma_Turnover_180d DESC
    LIMIT 15;
"""

df_clientes_consolidados = pd.read_sql(query_agregada, conexao)
print("Top 15 Clientes Consolidados por Movimentação Financeira:")
display(df_clientes_consolidados)


Top 15 Clientes Consolidados por Movimentação Financeira:


,case_id,Qtd_Transacoes_Registradas,Max_Saldo_180d,Soma_Turnover_180d,Soma_Turnover_30d,Data_Primeira_Abertura
0,1494474,33,449.59903,1.168074e+06,160762.230,2000-01-01 00:00:00
1,1500947,1,359.27835,9.000000e+05,0.000,2000-01-01 00:00:00
2,1319043,12,707.92500,5.477650e+05,53851.000,2000-01-01 00:00:00
3,1265705,12,707.92500,5.477650e+05,53851.000,2000-01-01 00:00:00
4,246503,32,1220.75900,5.182962e+05,55695.200,2000-01-01 00:00:00
5,151842,32,1220.75900,5.182962e+05,55695.200,2000-01-01 00:00:00
6,1579995,2,1671.48400,4.015988e+05,84147.400,2000-01-01 00:00:00
7,1543410,4,0.00000,3.910000e+05,390000.000,2000-01-01 00:00:00
8,244434,4,1317.72010,3.543495e+05,6665.000,2000-01-01 00:00:00
9,200453,3,0.00000,3.349600e+05,0.000,2000-01-01 00:00:00


In [ ]:
import pandas as pd
import sqlite3
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

conexao = sqlite3.connect('banco_risco_credito.db')

# 1. Carregando a base agregada completa do SQLite para o Pandas
query_completa = """
    SELECT
        case_id,
        Qtd_Transacoes_Registradas,
        Max_Saldo_180d,
        Soma_Turnover_180d,
        Soma_Turnover_30d
    FROM (
        SELECT
            case_id,
            COUNT(case_id) AS Qtd_Transacoes_Registradas,
            MAX(last180dayaveragebalance_704A) AS Max_Saldo_180d,
            SUM(last180dayturnover_1134A) AS Soma_Turnover_180d,
            SUM(last30dayturnover_651A) AS Soma_Turnover_30d
        FROM tb_credito_tratada
        GROUP BY case_id
    );
"""

df_ml = pd.read_sql(query_completa, conexao)

# 2. Selecionando as features (variáveis) para o modelo de Machine Learning
features = ['Qtd_Transacoes_Registradas', 'Max_Saldo_180d', 'Soma_Turnover_180d', 'Soma_Turnover_30d']
X = df_ml[features]

# 3. Padronizando os dados (Normalização essencial para algoritmos de ML)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Aplicando Machine Learning (K-Means para segmentar perfis de risco/movimentação em 3 clusters)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_ml['Cluster_Perfil_Risco'] = kmeans.fit_predict(X_scaled)

# Mapeando os clusters para termos de negócios claros
mapeamento_clusters = {0: 'Baixo Risco / Padrão', 1: 'Alto Volume / VIP', 2: 'Médio Risco / Ativo'}
df_ml['Perfil_Comercial'] = df_ml['Cluster_Perfil_Risco'].map(mapeamento_clusters)

# 5. Salvando a tabela final com os resultados do Machine Learning no SQLite
df_ml.to_sql('tb_resultado_ml_credito', conexao, if_exists='replace', index=False)

print("Modelo de Machine Learning executado com sucesso!")
print(df_ml['Perfil_Comercial'].value_counts())
print("\nAmostra dos clientes segmentados pelo modelo:")
display(df_ml[['case_id', 'Soma_Turnover_180d', 'Perfil_Comercial']].head(10))


Modelo de Machine Learning executado com sucesso!
Perfil_Comercial
Baixo Risco / Padrão    106900
Médio Risco / Ativo       4407
Alto Volume / VIP          465
Name: count, dtype: int64

Amostra dos clientes segmentados pelo modelo:


,case_id,Soma_Turnover_180d,Perfil_Comercial
0,225,0.0,Baixo Risco / Padrão
1,331,0.0,Baixo Risco / Padrão
2,358,0.0,Baixo Risco / Padrão
3,390,0.0,Baixo Risco / Padrão
4,445,0.0,Baixo Risco / Padrão
5,450,0.0,Baixo Risco / Padrão
6,453,0.0,Baixo Risco / Padrão
7,582,0.0,Baixo Risco / Padrão
8,649,0.0,Baixo Risco / Padrão
9,697,0.0,Baixo Risco / Padrão
